# Analysis on City Sprawl Through the Lens of Winnipeg Building Permits

## Cleaning notebook
Aylie Lapierre Nagler <br>
COMP 2040 <br>
April 21, 2026

The first step is to narrow down the possible analytical questions to provide cleaning guidance:

The insights I will be exploring in this data pertain to the patterns in the location of permits:
1. Are there hotspots for certain types of construction?
2. Are there any temporal patterns indicating the geographical direction of growth in the city?

Import modules:

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import your helper module
import sys
sys.path.insert(0, '..')
from src.helpers import *

Load data:

In [28]:
# Load data
df = pd.read_csv('../data/building-permits.csv', dtype={23: str, 24: str, 25: str, 26: str})

### Remove Time from *issue_date*

Issue date includes time which is always set to midnight. This is not meaningful and will be removed for clarity:

In [29]:
# Display column for demonstration of issue
df['issue_date'].head()

0    2010-02-01T00:00:00.000
1    2010-02-16T00:00:00.000
2    2010-03-19T00:00:00.000
3    2010-03-23T00:00:00.000
4    2010-04-13T00:00:00.000
Name: issue_date, dtype: object

In [30]:
# Remove time w/ remove_time() helper function
df['issue_date'] = remove_time(df['issue_date'])

In [31]:
# Confirm proper application
df['issue_date'].head()

0   2010-02-01
1   2010-02-16
2   2010-03-19
3   2010-03-23
4   2010-04-13
Name: issue_date, dtype: datetime64[ns]

### Remove time from *final_date*

Similar to *issue_date*, *final_date* includes messy time information that does not carry any meaning. As such, the time will be removed from this column as well:

In [32]:
# Display column for demonstration of issue
df['final_date'].head()

0    2012-01-11T00:00:00.000
1    2011-05-11T00:00:00.000
2    2010-10-27T00:00:00.000
3    2011-01-07T00:00:00.000
4    2011-07-28T00:00:00.000
Name: final_date, dtype: object

In [33]:
# Remove time w/ remove_time() helper function
df['final_date'] = remove_time(df['final_date'])

In [34]:
# Confirm proper application
df['final_date'].head()

0   2012-01-11
1   2011-05-11
2   2010-10-27
3   2011-01-07
4   2011-07-28
Name: final_date, dtype: datetime64[ns]

### Remove time from *application_received_date*

Similar to *issue_date* and *final_date*, *application_received_date* includes messy time information that does not carry any meaning. As such, the time will be removed from this column as well:

In [35]:
# Display column for demonstration of issue
df['application_received_date'].head()

0    2010-02-01T00:00:00.000
1    2010-02-16T00:00:00.000
2    2010-03-19T00:00:00.000
3    2010-03-23T00:00:00.000
4    2010-04-13T00:00:00.000
Name: application_received_date, dtype: object

In [36]:
# Remove time w/ remove_time() helper function
df['application_received_date'] = remove_time(df['application_received_date'])

In [37]:
# Confirm proper application
df['final_date'].head()

0   2012-01-11
1   2011-05-11
2   2010-10-27
3   2011-01-07
4   2011-07-28
Name: final_date, dtype: datetime64[ns]

#### Separate date columns into year, month, and day:

Dates will be separated into more atomic columns for further analysis

In [38]:
# Call on split_dates() function from helpers
df['issue_year'], df['issue_month'], df['issue_day'] = split_dates(df, 'issue_date')
df['final_year'], df['final_month'], df['final_day'] = split_dates(df, 'final_date')
df['application_received_year'], df['application_received_month'], df['application_received_day'] = split_dates(df, 'application_received_date')

#### Drop redundant date columns

In [39]:
df = df.drop(columns=['issue_date', 'final_date', 'application_received_date'])

### Assess whether filling in missing values in columns pertaining to address is worth it:

This data contains a complete *address* column, as well as *street_number*, *street_name*, *street_type*, and *street_direction* columns which store address data more atomically. The address column only has 3 missing values, whereas the more atomoic versions of it have a minimum of 272 missing values. If data in the *address* column is mostly stored in a consistent structure, it will be used to fill the missing values in other columns pertaining to address. If it is not stored in a consistent structure, geopy will be used to get coordinates instead:

In [40]:
# Use sample to get an idea of data structure
df['address'].sample(25)

122453            405 Waverley Street
135598          707 Prince Rupert AVE
76171                  790 Stella AVE
12540             83 Charlottetown RD
82595     1555 Regent AVE W Unit T76A
71908        895 Waverley St Unit 120
147842        5 Donald Street Unit300
137484              166 Scotia Street
58941                 679 Sargent AVE
114908           525 Pandora Avenue W
114975        509 Academy RD Unit 1-4
129497                  14 Acheson DR
78070     1600 Kenaston BLVD Unit 220
22618                1340 Dudley CRES
39897     104 Sandrington DR - Unit 2
101269              245 Vermillion RD
43847               6 Water Lily LANE
28061               22 & 22A Civic ST
82253                 100 Sunset BLVD
147213            287 Bertrand Street
46249                     542 Home ST
38302              54 Bridgeford PATH
45501               711 Kernaghan AVE
90735            600 Shaftesbury BLVD
53500                1177 Corydon AVE
Name: address, dtype: object

Observations:
- The first 'word' is always the street number
- The second word is always related to the street name
- Some street names are 2 (+?) words long, meaning the third word is not consistent
- The last word is either the street type or the unit number, with the second last word sometimes being 'UNIT' or 'Unit'
- Some rows have units that are names instead of numbers
- Some rows have dashes in the unit number
- Some of street types are all-caps abbreviations and others are the full word in title caps
- Some rows have 'Level' specified as the second last word, with the level number being the last row
- Some rows have street numbers that include a slash (e.g., 650/652 or 364 / 364)
- Some rows have 'building' and the building letter or number after the street type
- Some rows have an extra space (data is messy)

This data is not consistent at all. Neighbourhood data will be able to provide similar insights at less of a time cost. These columns will be dropped:

In [41]:
df = df.drop(columns=['street_number', 'street_name', 'street_type', 'street_direction'])

### Drop *x_coordinate_nad83* and *y_coordinate_nad83*

These columns containg data that is too microscopic to the geographic analysis I plan to do. Knowing the exact pinpoint of every permit issue is not important. Grouping by neighbourhoods is much more interpretable and relevant. These columns will be dropped.

In [42]:
df = df.drop(columns=['x_coordinate_nad83', 'y_coordinate_nad83'])

### Drop *point* column

*point* contains the same information as the above two coordinate columns (in a different unit). It will be dropped for the same reasons.

In [43]:
df = df.drop(columns='point')

### Drop *location*

Location contains the latitude and longitude coordinates paired up in tuple formatting. it will be deleted for the same reason as the previous 3 columns mentions.

In [44]:
df = df.drop(columns='location')

### Drop *ward* column

Wards can change, whereas neighbourhoods contain similar information but are much more stable. As such, wards will be dropped and neighbourhood_name will be kept:

In [45]:
df = df.drop(columns='ward')

### Drop *neighbourhood_number* column

Neighbourhood number does not carry any inherent meaning, and storing neighbourhoods numerically does not carry any computational value. neighbourhood_name will be kept instead:

In [46]:
df = df.drop(columns='neighbourhood_number')

### Drop *parent_permit_number* column

This is not relevant to my analysis questions which focus on temporal and spatial trends.

In [47]:
df = df.drop(columns='parent_permit_number')

### Investigate string suffix on *permit_number*

In [48]:
get_suffix(df['permit_number'])

{'AS',
 'CS',
 'EH',
 'HO',
 'MP',
 'MU',
 'OP',
 'PE',
 'PI',
 'PU',
 'RE',
 'RP',
 'SA',
 'SP',
 'TA',
 'TB'}

These likely denote the type of permit.:

In [49]:
df['permit_type'].unique()

array(['Housing', 'Multi-Residential', 'Accessory Structures',
       'Commercial Storage', 'Office', 'Health Services / Education',
       'Temporary Building Renewals', 'Sign', 'Retail',
       'Personal Services', 'Temporary Approval',
       'Recreation / Entertainment', 'General Public / Institutional',
       'Manufacturing', 'Public Utility', 'Safety / Protection'],
      dtype=object)

Many of the suffixes align, e.g.:
- HO: Housing
- CS: Commerical Storage
- PU: Public Utility <br><br>

These are communicating the same thing, however, dropping one or the other is not warranted since altering the suffix on *permit_number* would make it unsearchable in the full database, and without *permit_type* the suffixes would be meaningless. Both will be kept.

### Check for outliers

In [50]:
outliers(df)

,dwelling_units_created,dwelling_units_lost
count,29899.000000,3703.000000
percentage,18.932045,2.344739


The series pertaining to dwellings created/lost mostly contain values of 0. This function uses the IQR method to find outliers, which assumes a normal distribution, which these columns do not have. The outliers here are not *true* outliers in that they are not cause for concern or outside of expected values. As for latitude and longitude, the percentage of outliers is less than 0.5% which is basically negligible. None of these are cause for concern.

### Final columns after cleaning

In [51]:
# Get columns and data type in table format (2 cols)
split_columns(df)


,Column Names,Data Types,Column Names (cont.),Data Types (cont.)
0,permit_number,object,pool_type,object
1,permit_group,object,type_of_structure,object
2,permit_type,object,status,object
3,sub_type,object,economic_development_category,object
4,work_type,object,major_project,object
5,unit_type,object,address,object
6,unit_number,object,issue_year,int32
7,neighbourhood_name,object,issue_month,int32
8,community,object,issue_day,int32
9,applicant_business_name,object,final_year,float64


### Save cleaned dataframe to CSV

In [52]:
df.to_csv('../data/building-permits-clean.csv', index=False)